In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/README.txt
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/cvind.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2912.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2011.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/3027.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1912.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/3023.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/101.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1194.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/851.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2106.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1850.mat
/

In [4]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for f in files[:5]:
        print("  " * (depth+1) + f)
    if depth > 3:
        dirs[:] = []

    input/
      datasets/
        ashkhagan/


In [5]:

!find /kaggle/input -type f | head -50

/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/README.txt
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2912.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2011.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/3027.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1912.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/3023.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/101.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1194.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/851.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2106.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/1850.mat
/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data/2911.m

In [6]:
"""
Step 1: Pretrained-backbone baseline for brain tumor classification.

Replaces the paper's from-scratch 4-conv CNN with a fine-tuned EfficientNetB0,
adds BatchNorm, and fixes the train/val/test split to be patient-level
(using cjdata.PID from the .mat files) to avoid slice-level leakage.

Run on Kaggle with the figshare "brain_tumor_dataset" (.mat files) attached
as an input dataset. Set DATA_DIR below to that input path.
"""

import os
import glob
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data"
IMG_SIZE = 224                                     # EfficientNetB0 native size
BATCH_SIZE = 16
HEAD_EPOCHS = 10        # stage 1: train head only, backbone frozen
FINE_TUNE_EPOCHS = 40   # stage 2: unfreeze top of backbone, fine-tune end-to-end
FINE_TUNE_AT = 0.5      # unfreeze the top 50% of backbone layers
SEED = 42
NUM_CLASSES = 3
LABEL_MAP = {1: "meningioma", 2: "glioma", 3: "pituitary"}  # cjdata.label convention


# ---------------------------------------------------------------------------
# 1. Load .mat files -> images, labels, patient IDs
# ---------------------------------------------------------------------------
def load_dataset(data_dir):
    mat_files = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found under {data_dir}")

    images, labels, pids = [], [], []
    for f in mat_files:
        with h5py.File(f, "r") as h:
            cjdata = h["cjdata"]
            img = np.array(cjdata["image"]).astype(np.float32)
            label = int(np.array(cjdata["label"]).item())
            pid = np.array(cjdata["PID"])
            pid = "".join(chr(c) for c in pid.flatten()) if pid.dtype.kind in "uS" else str(int(pid.item()))

            images.append(img)
            labels.append(label - 1)  # -> 0,1,2
            pids.append(pid)

    return images, np.array(labels), np.array(pids)


def preprocess(img):
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = tf.image.resize(img[..., None], [IMG_SIZE, IMG_SIZE]).numpy()
    img = np.repeat(img, 3, axis=-1)  # grayscale -> 3-channel for ImageNet backbone
    return (img * 255).astype(np.uint8)


# ---------------------------------------------------------------------------
# 2. Patient-level stratified split
# ---------------------------------------------------------------------------
def patient_level_split(images, labels, pids, seed=SEED):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(images, labels, groups=pids))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=seed)  # 0.125*0.8=0.1 -> 70/10/20
    train_idx, val_idx = next(gss2.split(
        [images[i] for i in trainval_idx],
        labels[trainval_idx],
        groups=pids[trainval_idx],
    ))
    train_idx = trainval_idx[train_idx]
    val_idx = trainval_idx[val_idx]

    assert set(pids[train_idx]).isdisjoint(pids[val_idx])
    assert set(pids[train_idx]).isdisjoint(pids[test_idx])
    assert set(pids[val_idx]).isdisjoint(pids[test_idx])

    return train_idx, val_idx, test_idx


# ---------------------------------------------------------------------------
# 3. Model: EfficientNetB0 backbone + BatchNorm head
# ---------------------------------------------------------------------------
def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model, backbone


def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])


# ---------------------------------------------------------------------------
# 4. Train
# ---------------------------------------------------------------------------
def main():
    print("Loading dataset...")
    images_raw, labels, pids = load_dataset(DATA_DIR)
    print(f"Loaded {len(images_raw)} slices from {len(set(pids))} patients")

    train_idx, val_idx, test_idx = patient_level_split(images_raw, labels, pids)
    print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

    print("Preprocessing images (resize/normalize)...")
    X = np.stack([preprocess(images_raw[i]) for i in range(len(images_raw))])
    y = labels

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    augmenter = make_augmenter()

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
    test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
               .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model, backbone = build_model()

    # --- Stage 1: train head only, backbone frozen ---
    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("\n=== Stage 1: training head (backbone frozen) ===")
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS)

    # --- Stage 2: unfreeze top of backbone, fine-tune end-to-end ---
    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
        callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
    ]
    print("\n=== Stage 2: fine-tuning top of backbone ===")
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=cbs)

    # --- Evaluate ---
    print("\n=== Test set evaluation ===")
    y_pred = np.argmax(model.predict(test_ds), axis=1)
    print(classification_report(y_test, y_pred, target_names=list(LABEL_MAP.values())))
    print(confusion_matrix(y_test, y_pred))


if __name__ == "__main__":
    main()


Loading dataset...
Loaded 3064 slices from 233 patients
Train: 2090  Val: 362  Test: 612
Preprocessing images (resize/normalize)...


I0000 00:00:1790059021.098752      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790059021.101650      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== Stage 1: training head (backbone frozen) ===
Epoch 1/10


2026-09-22 06:37:33.939783: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:37:34.080358: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:37:34.400637: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:37:34.543175: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:37:34.684278: E external/local_xla/xla/stream_

130/131 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.7130 - loss: 0.8303

...

130/131 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.6224 - loss: 0.9532

2026-09-22 06:41:37.209337: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:41:37.346929: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:41:37.952535: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:41:38.089219: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


131/131 ━━━━━━━━━━━━━━━━━━━━ 96s 383ms/step - accuracy: 0.6617 - loss: 0.8091 - val_accuracy: 0.8177 - val_loss: 0.4592 - learning_rate: 1.0000e-05
Epoch 2/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 18s 134ms/step - accuracy: 0.7330 - loss: 0.6430 - val_accuracy: 0.8370 - val_loss: 0.4459 - learning_rate: 1.0000e-05
Epoch 3/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 131ms/step - accuracy: 0.7799 - loss: 0.5558 - val_accuracy: 0.8481 - val_loss: 0.3996 - learning_rate: 1.0000e-05
Epoch 4/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 131ms/step - accuracy: 0.7876 - loss: 0.4980 - val_accuracy: 0.8646 - val_loss: 0.3708 - learning_rate: 1.0000e-05
Epoch 5/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 132ms/step - accuracy: 0.8005 - loss: 0.4779 - val_accuracy: 0.8702 - val_loss: 0.3430 - learning_rate: 1.0000e-05
Epoch 6/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 132ms/step - accuracy: 0.8091 - loss: 0.4767 - val_accuracy: 0.8729 - val_loss: 0.3231 - learning_rate: 1.0000e-05
Epoch 7/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 132ms/step - 

2026-09-22 06:53:17.628280: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:53:17.763224: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:53:18.083050: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:53:18.223521: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 06:53:19.002123: E external/local_xla/xla/stream_

39/39 ━━━━━━━━━━━━━━━━━━━━ 19s 339ms/step
              precision    recall  f1-score   support

  meningioma       0.77      0.80      0.78        98
      glioma       0.97      0.91      0.94       288
   pituitary       0.93      0.98      0.95       226

    accuracy                           0.92       612
   macro avg       0.89      0.90      0.89       612
weighted avg       0.92      0.92      0.92       612

[[ 78   7  13]
 [ 20 263   5]
 [  3   1 222]]


In [8]:
"""
Step 1+2: Pretrained-backbone + mask-guided ROI cropping for brain tumor classification.

Improvements over the paper's from-scratch 4-conv CNN + RF:
  - EfficientNetB0 backbone, ImageNet-pretrained, two-stage fine-tuning
  - BatchNorm in the head (missing in the original)
  - Patient-level train/val/test split (using cjdata.PID) -> no leakage
  - Class-weighted loss (meningioma is underrepresented and hardest class)
  - Mask-guided ROI cropping: crop around the tumor mask (with margin) before
    resizing, instead of feeding the whole 512x512 slice with skull/background.
    This directly targets the meningioma<->pituitary/glioma confusion seen in
    the whole-slice baseline, since meningioma sits near the skull and gets
    confused with surrounding structure when the model sees the full image.
  - Test-time augmentation (TTA) at inference for a free accuracy bump.

Run on Kaggle with GPU enabled. DATA_DIR points at the figshare .mat files.
"""

import os
import glob
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data"
IMG_SIZE = 224                                     # EfficientNetB0 native size
BATCH_SIZE = 16
HEAD_EPOCHS = 10          # stage 1: train head only, backbone frozen
FINE_TUNE_EPOCHS = 60     # stage 2: unfreeze top of backbone, fine-tune end-to-end
FINE_TUNE_AT = 0.8        # unfreeze the top 80% of backbone layers (was 0.5)
CROP_MARGIN = 0.25        # fraction of tumor bbox size added as margin on each side
TTA_ROUNDS = 8            # number of augmented views averaged at test time
SEED = 42
NUM_CLASSES = 3
LABEL_MAP = {1: "meningioma", 2: "glioma", 3: "pituitary"}  # cjdata.label convention


# ---------------------------------------------------------------------------
# 1. Load .mat files -> images, masks, labels, patient IDs
# ---------------------------------------------------------------------------
def load_dataset(data_dir):
    mat_files = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found under {data_dir}")

    images, masks, labels, pids = [], [], [], []
    for f in mat_files:
        with h5py.File(f, "r") as h:
            cjdata = h["cjdata"]
            img = np.array(cjdata["image"]).astype(np.float32)
            mask = np.array(cjdata["tumorMask"]).astype(np.float32)
            label = int(np.array(cjdata["label"]).item())
            pid = np.array(cjdata["PID"])
            pid = "".join(chr(c) for c in pid.flatten()) if pid.dtype.kind in "uS" else str(int(pid.item()))

            images.append(img)
            masks.append(mask)
            labels.append(label - 1)  # -> 0,1,2
            pids.append(pid)

    return images, masks, np.array(labels), np.array(pids)


def crop_to_mask(img, mask, margin=CROP_MARGIN):
    """Crop image to the tumor mask's bounding box, with a margin, so the
    model sees tumor + local context instead of the whole brain + skull."""
    ys, xs = np.where(mask > 0.5)
    if len(ys) == 0:
        return img  # no mask -> fall back to full image
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    h, w = y1 - y0, x1 - x0
    pad_y, pad_x = int(h * margin) + 1, int(w * margin) + 1
    y0 = max(0, y0 - pad_y)
    y1 = min(img.shape[0], y1 + pad_y)
    x0 = max(0, x0 - pad_x)
    x1 = min(img.shape[1], x1 + pad_x)
    return img[y0:y1, x0:x1]


def preprocess(img, mask):
    img = crop_to_mask(img, mask)
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = tf.image.resize(img[..., None], [IMG_SIZE, IMG_SIZE]).numpy()
    img = np.repeat(img, 3, axis=-1)  # grayscale -> 3-channel for ImageNet backbone
    return (img * 255).astype(np.uint8)


# ---------------------------------------------------------------------------
# 2. Patient-level stratified split
# ---------------------------------------------------------------------------
def patient_level_split(images, labels, pids, seed=SEED):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(images, labels, groups=pids))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=seed)  # 0.125*0.8=0.1 -> 70/10/20
    train_idx, val_idx = next(gss2.split(
        [images[i] for i in trainval_idx],
        labels[trainval_idx],
        groups=pids[trainval_idx],
    ))
    train_idx = trainval_idx[train_idx]
    val_idx = trainval_idx[val_idx]

    assert set(pids[train_idx]).isdisjoint(pids[val_idx])
    assert set(pids[train_idx]).isdisjoint(pids[test_idx])
    assert set(pids[val_idx]).isdisjoint(pids[test_idx])

    return train_idx, val_idx, test_idx


# ---------------------------------------------------------------------------
# 3. Model: EfficientNetB0 backbone + BatchNorm head
# ---------------------------------------------------------------------------
def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model, backbone


def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])


def tta_predict(model, augmenter, X, rounds=TTA_ROUNDS, batch_size=BATCH_SIZE):
    """Average predictions over several augmented views of each test image."""
    probs = np.zeros((X.shape[0], NUM_CLASSES), dtype=np.float32)
    # one clean (non-augmented) pass
    probs += model.predict(X, batch_size=batch_size, verbose=0)
    for _ in range(rounds):
        X_aug = augmenter(X, training=True).numpy()
        probs += model.predict(X_aug, batch_size=batch_size, verbose=0)
    probs /= (rounds + 1)
    return probs


# ---------------------------------------------------------------------------
# 4. Train
# ---------------------------------------------------------------------------
def main():
    print("Loading dataset...")
    images_raw, masks_raw, labels, pids = load_dataset(DATA_DIR)
    print(f"Loaded {len(images_raw)} slices from {len(set(pids))} patients")

    train_idx, val_idx, test_idx = patient_level_split(images_raw, labels, pids)
    print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

    print("Preprocessing images (mask-guided crop + resize/normalize)...")
    X = np.stack([preprocess(images_raw[i], masks_raw[i]) for i in range(len(images_raw))])
    y = labels

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    augmenter = make_augmenter()

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model, backbone = build_model()

    class_weight = dict(enumerate(
        compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
    ))
    print(f"Class weights {{meningioma, glioma, pituitary}}: {class_weight}")

    # --- Stage 1: train head only, backbone frozen ---
    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("\n=== Stage 1: training head (backbone frozen) ===")
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weight)

    # --- Stage 2: unfreeze top of backbone, fine-tune end-to-end ---
    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
        callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
    ]
    print("\n=== Stage 2: fine-tuning top of backbone ===")
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=cbs, class_weight=class_weight)

    # --- Evaluate: plain single-pass prediction ---
    print("\n=== Test set evaluation (single pass) ===")
    y_pred_plain = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE), axis=1)
    print(classification_report(y_test, y_pred_plain, target_names=list(LABEL_MAP.values())))
    print(confusion_matrix(y_test, y_pred_plain))

    # --- Evaluate: test-time augmentation ---
    print(f"\n=== Test set evaluation (TTA, {TTA_ROUNDS} rounds) ===")
    probs_tta = tta_predict(model, augmenter, X_test)
    y_pred_tta = np.argmax(probs_tta, axis=1)
    print(classification_report(y_test, y_pred_tta, target_names=list(LABEL_MAP.values())))
    print(confusion_matrix(y_test, y_pred_tta))


if __name__ == "__main__":
    main()


Loading dataset...
Loaded 3064 slices from 233 patients
Train: 2090  Val: 362  Test: 612
Preprocessing images (mask-guided crop + resize/normalize)...
Class weights {meningioma, glioma, pituitary}: {0: np.float64(1.2179487179487178), 1: np.float64(0.6959706959706959), 2: np.float64(1.3475177304964538)}

=== Stage 1: training head (backbone frozen) ===
Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 57s 261ms/step - accuracy: 0.7220 - loss: 0.8099 - val_accuracy: 0.7044 - val_loss: 0.6844
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.7890 - loss: 0.5650 - val_accuracy: 0.7707 - val_loss: 0.5699
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 13s 96ms/step - accuracy: 0.8057 - loss: 0.5234 - val_accuracy: 0.7624 - val_loss: 0.6150
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 13s 96ms/step - accuracy: 0.8330 - loss: 0.4455 - val_accuracy: 0.7790 - val_loss: 0.5713
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 13s 95ms/step - accuracy: 0.8373 - loss: 0.4264 - val_accuracy: 0.7376 - val_loss: 0.

In [ ]:
"""
Step 3: Patient-level 5-fold cross-validation + CNN-embedding ensemble classifier.

Builds on step1_baseline.py (EfficientNetB0 backbone, mask-guided ROI cropping,
class weighting, TTA). This script adds:

  - 5-fold patient-level GroupKFold CV, so every patient is held out exactly
    once -> gives a mean +/- std accuracy instead of one noisy single-split
    number. This is the statistically defensible way to report results for
    a journal, especially with only 233 patients.
  - Per-fold: train the CNN, then extract its penultimate-layer embedding and
    fit a soft-voting ensemble (Random Forest + XGBoost + SVM) on top of it,
    compared against the CNN's own softmax head (with and without TTA).

Run on Kaggle with GPU enabled. This trains 5 models sequentially -- budget
several hours. If Kaggle's 9-12h session limit is a concern, reduce N_FOLDS
or FINE_TUNE_EPOCHS below.
"""

import os
import glob
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data"
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 35
FINE_TUNE_AT = 0.8
CROP_MARGIN = 0.25
TTA_ROUNDS = 5
N_FOLDS = 5
SEED = 42
NUM_CLASSES = 3
LABEL_MAP = {1: "meningioma", 2: "glioma", 3: "pituitary"}


# ---------------------------------------------------------------------------
# 1. Load .mat files -> images, masks, labels, patient IDs
# ---------------------------------------------------------------------------
def load_dataset(data_dir):
    mat_files = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found under {data_dir}")

    images, masks, labels, pids = [], [], [], []
    for f in mat_files:
        with h5py.File(f, "r") as h:
            cjdata = h["cjdata"]
            img = np.array(cjdata["image"]).astype(np.float32)
            mask = np.array(cjdata["tumorMask"]).astype(np.float32)
            label = int(np.array(cjdata["label"]).item())
            pid = np.array(cjdata["PID"])
            pid = "".join(chr(c) for c in pid.flatten()) if pid.dtype.kind in "uS" else str(int(pid.item()))

            images.append(img)
            masks.append(mask)
            labels.append(label - 1)
            pids.append(pid)

    return images, masks, np.array(labels), np.array(pids)


def crop_to_mask(img, mask, margin=CROP_MARGIN):
    ys, xs = np.where(mask > 0.5)
    if len(ys) == 0:
        return img
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    h, w = y1 - y0, x1 - x0
    pad_y, pad_x = int(h * margin) + 1, int(w * margin) + 1
    y0 = max(0, y0 - pad_y)
    y1 = min(img.shape[0], y1 + pad_y)
    x0 = max(0, x0 - pad_x)
    x1 = min(img.shape[1], x1 + pad_x)
    return img[y0:y1, x0:x1]


def preprocess(img, mask):
    img = crop_to_mask(img, mask)
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = tf.image.resize(img[..., None], [IMG_SIZE, IMG_SIZE]).numpy()
    img = np.repeat(img, 3, axis=-1)
    return (img * 255).astype(np.uint8)


# ---------------------------------------------------------------------------
# 2. Model: EfficientNetB0 backbone, with an embedding-extraction hook
# ---------------------------------------------------------------------------
def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    embedding = layers.Dense(128, activation="relu", name="embedding")(x)
    x = layers.BatchNormalization()(embedding)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    embedding_model = models.Model(inputs, embedding)
    return model, backbone, embedding_model


def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])


def tta_predict(model, augmenter, X, rounds=TTA_ROUNDS, batch_size=BATCH_SIZE):
    probs = np.zeros((X.shape[0], NUM_CLASSES), dtype=np.float32)
    probs += model.predict(X, batch_size=batch_size, verbose=0)
    for _ in range(rounds):
        X_aug = augmenter(X, training=True).numpy()
        probs += model.predict(X_aug, batch_size=batch_size, verbose=0)
    probs /= (rounds + 1)
    return probs


def train_fold_model(X_train, y_train, X_val, y_val, class_weight, augmenter):
    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model, backbone, embedding_model = build_model()

    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              class_weight=class_weight, verbose=2)

    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=cbs, class_weight=class_weight, verbose=2)

    return model, embedding_model


def fit_ensemble(embeddings_train, y_train):
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight="balanced")
    xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                         random_state=SEED, eval_metric="mlogloss")
    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)

    ensemble = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb), ("svm", svm)],
        voting="soft",
    )
    ensemble.fit(embeddings_train, y_train)
    return ensemble


# ---------------------------------------------------------------------------
# 3. Cross-validation driver
# ---------------------------------------------------------------------------
def main():
    print("Loading dataset...")
    images_raw, masks_raw, labels, pids = load_dataset(DATA_DIR)
    print(f"Loaded {len(images_raw)} slices from {len(set(pids))} patients")

    print("Preprocessing images (mask-guided crop + resize/normalize)...")
    X = np.stack([preprocess(images_raw[i], masks_raw[i]) for i in range(len(images_raw))])
    y = labels

    unique_pids = np.array(sorted(set(pids)))
    gkf = GroupKFold(n_splits=N_FOLDS)

    fold_results = {"cnn": [], "cnn_tta": [], "ensemble": []}
    all_val_frac = 0.15

    for fold_idx, (trainval_idx, test_idx) in enumerate(gkf.split(X, y, groups=pids)):
        print(f"\n{'='*70}\nFOLD {fold_idx + 1}/{N_FOLDS}\n{'='*70}")

        gss = GroupShuffleSplit(n_splits=1, test_size=all_val_frac, random_state=SEED)
        tr_sub, val_sub = next(gss.split(
            trainval_idx, y[trainval_idx], groups=pids[trainval_idx]
        ))
        train_idx = trainval_idx[tr_sub]
        val_idx = trainval_idx[val_sub]

        assert set(pids[train_idx]).isdisjoint(pids[test_idx])
        assert set(pids[val_idx]).isdisjoint(pids[test_idx])
        assert set(pids[train_idx]).isdisjoint(pids[val_idx])

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

        class_weight = dict(enumerate(
            compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
        ))
        augmenter = make_augmenter()

        model, embedding_model = train_fold_model(
            X_train, y_train, X_val, y_val, class_weight, augmenter
        )

        # --- CNN softmax, single pass ---
        y_pred_cnn = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
        acc_cnn = accuracy_score(y_test, y_pred_cnn)
        fold_results["cnn"].append(acc_cnn)

        # --- CNN softmax + TTA ---
        probs_tta = tta_predict(model, augmenter, X_test)
        y_pred_tta = np.argmax(probs_tta, axis=1)
        acc_tta = accuracy_score(y_test, y_pred_tta)
        fold_results["cnn_tta"].append(acc_tta)

        # --- Embedding ensemble (RF + XGBoost + SVM) ---
        emb_train = embedding_model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
        emb_test = embedding_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
        ensemble = fit_ensemble(emb_train, y_train)
        y_pred_ens = ensemble.predict(emb_test)
        acc_ens = accuracy_score(y_test, y_pred_ens)
        fold_results["ensemble"].append(acc_ens)

        print(f"\nFold {fold_idx + 1} results: CNN={acc_cnn:.4f}  CNN+TTA={acc_tta:.4f}  Ensemble={acc_ens:.4f}")
        print("\n-- CNN+TTA classification report --")
        print(classification_report(y_test, y_pred_tta, target_names=list(LABEL_MAP.values())))
        print(confusion_matrix(y_test, y_pred_tta))
        print("\n-- Embedding ensemble classification report --")
        print(classification_report(y_test, y_pred_ens, target_names=list(LABEL_MAP.values())))
        print(confusion_matrix(y_test, y_pred_ens))

        tf.keras.backend.clear_session()

    print(f"\n{'='*70}\nFINAL 5-FOLD CROSS-VALIDATION SUMMARY\n{'='*70}")
    for key, label in [("cnn", "CNN (softmax)"), ("cnn_tta", "CNN + TTA"), ("ensemble", "CNN-embedding ensemble (RF+XGB+SVM)")]:
        accs = np.array(fold_results[key])
        print(f"{label:40s}: {accs.mean()*100:.2f}% +/- {accs.std()*100:.2f}%   (folds: {[f'{a*100:.2f}' for a in accs]})")


if __name__ == "__main__":
    main()


Loading dataset...
Loaded 3064 slices from 233 patients
Preprocessing images (mask-guided crop + resize/normalize)...

FOLD 1/5
Train: 2012  Val: 439  Test: 613
Epoch 1/8


2026-09-22 08:39:07.931656: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:39:08.070660: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:39:08.397399: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:39:08.539797: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:39:09.325679: E external/local_xla/xla/stream_

126/126 - 67s - 534ms/step - accuracy: 0.7097 - loss: 0.8072 - val_accuracy: 0.8360 - val_loss: 0.4382
Epoch 2/8
126/126 - 13s - 101ms/step - accuracy: 0.7828 - loss: 0.5612 - val_accuracy: 0.8702 - val_loss: 0.3359
Epoch 3/8
126/126 - 13s - 102ms/step - accuracy: 0.8156 - loss: 0.4852 - val_accuracy: 0.9180 - val_loss: 0.2215
Epoch 4/8
126/126 - 13s - 100ms/step - accuracy: 0.8280 - loss: 0.4638 - val_accuracy: 0.9226 - val_loss: 0.2171
Epoch 5/8
126/126 - 13s - 100ms/step - accuracy: 0.8429 - loss: 0.3932 - val_accuracy: 0.9180 - val_loss: 0.2054
Epoch 6/8
126/126 - 13s - 99ms/step - accuracy: 0.8534 - loss: 0.3968 - val_accuracy: 0.9066 - val_loss: 0.2365
Epoch 7/8
126/126 - 13s - 100ms/step - accuracy: 0.8499 - loss: 0.3895 - val_accuracy: 0.9180 - val_loss: 0.2130
Epoch 8/8
126/126 - 13s - 101ms/step - accuracy: 0.8405 - loss: 0.4109 - val_accuracy: 0.9226 - val_loss: 0.2097
Epoch 1/35


2026-09-22 08:42:26.284876: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:42:26.426340: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:42:28.351117: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:42:28.490066: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


126/126 - 118s - 939ms/step - accuracy: 0.6014 - loss: 1.1207 - val_accuracy: 0.8542 - val_loss: 0.3640 - learning_rate: 1.0000e-05
Epoch 2/35
126/126 - 13s - 104ms/step - accuracy: 0.6750 - loss: 0.8496 - val_accuracy: 0.8497 - val_loss: 0.3743 - learning_rate: 1.0000e-05
Epoch 3/35
126/126 - 13s - 105ms/step - accuracy: 0.7162 - loss: 0.7958 - val_accuracy: 0.8314 - val_loss: 0.4006 - learning_rate: 1.0000e-05
Epoch 4/35
126/126 - 13s - 105ms/step - accuracy: 0.7560 - loss: 0.6565 - val_accuracy: 0.8337 - val_loss: 0.3862 - learning_rate: 1.0000e-05
Epoch 5/35
126/126 - 13s - 105ms/step - accuracy: 0.7674 - loss: 0.6136 - val_accuracy: 0.8474 - val_loss: 0.3374 - learning_rate: 1.0000e-05
Epoch 6/35
126/126 - 13s - 106ms/step - accuracy: 0.7848 - loss: 0.5509 - val_accuracy: 0.8610 - val_loss: 0.3107 - learning_rate: 1.0000e-05
Epoch 7/35
126/126 - 13s - 105ms/step - accuracy: 0.7972 - loss: 0.5253 - val_accuracy: 0.8702 - val_loss: 0.2933 - learning_rate: 1.0000e-05
Epoch 8/35
126/1

...


Fold 1 results: CNN=0.9005  CNN+TTA=0.9086  Ensemble=0.8923

-- CNN+TTA classification report --
              precision    recall  f1-score   support

  meningioma       0.83      0.81      0.82       125
      glioma       0.99      0.96      0.97       239
   pituitary       0.87      0.91      0.89       249

    accuracy                           0.91       613
   macro avg       0.90      0.89      0.89       613
weighted avg       0.91      0.91      0.91       613

[[101   0  24]
 [  1 229   9]
 [ 20   2 227]]

-- Embedding ensemble classification report --
              precision    recall  f1-score   support

  meningioma       0.82      0.77      0.79       125
      glioma       0.95      0.96      0.95       239
   pituitary       0.87      0.89      0.88       249

    accuracy                           0.89       613
   macro avg       0.88      0.87      0.88       613
weighted avg       0.89      0.89      0.89       613

[[ 96   5  24]
 [  1 230   8]
 [ 20   8 221]]


2026-09-22 08:55:37.873029: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:55:38.010405: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:55:39.840456: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 08:55:39.976479: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


128/128 - 118s - 925ms/step - accuracy: 0.5679 - loss: 1.2090 - val_accuracy: 0.7888 - val_loss: 0.4704 - learning_rate: 1.0000e-05
Epoch 2/35
128/128 - 13s - 104ms/step - accuracy: 0.6405 - loss: 0.9304 - val_accuracy: 0.7913 - val_loss: 0.4696 - learning_rate: 1.0000e-05
Epoch 3/35
128/128 - 13s - 104ms/step - accuracy: 0.6940 - loss: 0.7904 - val_accuracy: 0.8301 - val_loss: 0.4264 - learning_rate: 1.0000e-05
Epoch 4/35
128/128 - 13s - 104ms/step - accuracy: 0.7376 - loss: 0.6828 - val_accuracy: 0.8495 - val_loss: 0.4011 - learning_rate: 1.0000e-05
Epoch 5/35
128/128 - 13s - 104ms/step - accuracy: 0.7450 - loss: 0.6448 - val_accuracy: 0.8568 - val_loss: 0.3720 - learning_rate: 1.0000e-05
Epoch 6/35
128/128 - 13s - 104ms/step - accuracy: 0.7803 - loss: 0.5982 - val_accuracy: 0.8665 - val_loss: 0.3509 - learning_rate: 1.0000e-05
Epoch 7/35
128/128 - 13s - 104ms/step - accuracy: 0.7818 - loss: 0.5417 - val_accuracy: 0.8689 - val_loss: 0.3325 - learning_rate: 1.0000e-05
Epoch 8/35
128/1

2026-09-22 09:06:04.093496: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 09:06:04.231183: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 09:06:04.554333: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 09:06:04.696205: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-22 09:06:05.474432: E external/local_xla/xla/stream_

135/135 - 61s - 453ms/step - accuracy: 0.7238 - loss: 0.7646 - val_accuracy: 0.8418 - val_loss: 0.4365
Epoch 2/8
135/135 - 13s - 98ms/step - accuracy: 0.7846 - loss: 0.5775 - val_accuracy: 0.8418 - val_loss: 0.3405
Epoch 3/8
135/135 - 13s - 98ms/step - accuracy: 0.8092 - loss: 0.5081 - val_accuracy: 0.8519 - val_loss: 0.3354
Epoch 4/8
135/135 - 13s - 99ms/step - accuracy: 0.8194 - loss: 0.4728 - val_accuracy: 0.8956 - val_loss: 0.2584
Epoch 5/8
135/135 - 13s - 100ms/step - accuracy: 0.8422 - loss: 0.3934 - val_accuracy: 0.8855 - val_loss: 0.2801
Epoch 6/8
135/135 - 13s - 99ms/step - accuracy: 0.8403 - loss: 0.4303 - val_accuracy: 0.8956 - val_loss: 0.2641
Epoch 7/8
135/135 - 13s - 99ms/step - accuracy: 0.8435 - loss: 0.4084 - val_accuracy: 0.8990 - val_loss: 0.2490
Epoch 8/8
135/135 - 13s - 98ms/step - accuracy: 0.8533 - loss: 0.3776 - val_accuracy: 0.8822 - val_loss: 0.2476
Epoch 1/35
135/135 - 116s - 860ms/step - accuracy: 0.5947 - loss: 1.1390 - val_accuracy: 0.8316 - val_loss: 0.45

In [2]:
"""
Step 3: Patient-level 5-fold cross-validation + CNN-embedding ensemble classifier.

Builds on step1_baseline.py (EfficientNetB0 backbone, mask-guided ROI cropping,
class weighting, TTA). This script adds:

  - 5-fold patient-level GroupKFold CV, so every patient is held out exactly
    once -> gives a mean +/- std accuracy instead of one noisy single-split
    number. This is the statistically defensible way to report results for
    a journal, especially with only 233 patients.
  - Per-fold: train the CNN, then extract its penultimate-layer embedding and
    fit a soft-voting ensemble (Random Forest + XGBoost + SVM) on top of it,
    compared against the CNN's own softmax head (with and without TTA).

Run on Kaggle with GPU enabled. This trains 5 models sequentially -- budget
several hours. If Kaggle's 9-12h session limit is a concern, reduce N_FOLDS
or FINE_TUNE_EPOCHS below.
"""

import os
import gc
import glob
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset/dataset/data"
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 35
FINE_TUNE_AT = 0.8
CROP_MARGIN = 0.25
TTA_ROUNDS = 5
N_FOLDS = 5
SEED = 42
NUM_CLASSES = 3
LABEL_MAP = {1: "meningioma", 2: "glioma", 3: "pituitary"}


def crop_to_mask(img, mask, margin=CROP_MARGIN):
    ys, xs = np.where(mask > 0.5)
    if len(ys) == 0:
        return img
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    h, w = y1 - y0, x1 - x0
    pad_y, pad_x = int(h * margin) + 1, int(w * margin) + 1
    y0 = max(0, y0 - pad_y)
    y1 = min(img.shape[0], y1 + pad_y)
    x0 = max(0, x0 - pad_x)
    x1 = min(img.shape[1], x1 + pad_x)
    return img[y0:y1, x0:x1]


def preprocess(img, mask):
    img = crop_to_mask(img, mask)
    img = img - img.min()
    img = img / (img.max() + 1e-8)
    img = tf.image.resize(img[..., None], [IMG_SIZE, IMG_SIZE]).numpy()
    img = np.repeat(img, 3, axis=-1)
    return (img * 255).astype(np.uint8)


def load_and_preprocess_dataset(data_dir):
    """Load + crop/resize each .mat file one at a time and write straight into
    the final uint8 array, instead of holding all 3000+ raw float32 images and
    masks in memory at once (~6GB peak) before preprocessing them. Same
    preprocessing per file, same file order -> identical output to the
    two-pass version, just without the memory spike that was causing the
    Kaggle notebook to OOM and restart."""
    mat_files = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found under {data_dir}")

    n = len(mat_files)
    X = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    labels = np.zeros(n, dtype=np.int64)
    pids = []

    for i, f in enumerate(mat_files):
        with h5py.File(f, "r") as h:
            cjdata = h["cjdata"]
            img = np.array(cjdata["image"]).astype(np.float32)
            mask = np.array(cjdata["tumorMask"]).astype(np.float32)
            label = int(np.array(cjdata["label"]).item())
            pid = np.array(cjdata["PID"])
            pid = "".join(chr(c) for c in pid.flatten()) if pid.dtype.kind in "uS" else str(int(pid.item()))

        X[i] = preprocess(img, mask)
        labels[i] = label - 1
        pids.append(pid)
        del img, mask

        if (i + 1) % 500 == 0 or (i + 1) == n:
            print(f"  processed {i + 1}/{n}")

    return X, labels, np.array(pids)


# ---------------------------------------------------------------------------
# 2. Model: EfficientNetB0 backbone, with an embedding-extraction hook
# ---------------------------------------------------------------------------
def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    embedding = layers.Dense(128, activation="relu", name="embedding")(x)
    x = layers.BatchNormalization()(embedding)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    embedding_model = models.Model(inputs, embedding)
    return model, backbone, embedding_model


def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])


def tta_predict(model, augmenter, X, rounds=TTA_ROUNDS, batch_size=BATCH_SIZE):
    probs = np.zeros((X.shape[0], NUM_CLASSES), dtype=np.float32)
    probs += model.predict(X, batch_size=batch_size, verbose=0)
    for _ in range(rounds):
        X_aug = augmenter(X, training=True).numpy()
        probs += model.predict(X_aug, batch_size=batch_size, verbose=0)
    probs /= (rounds + 1)
    return probs


def train_fold_model(X_train, y_train, X_val, y_val, class_weight, augmenter):
    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model, backbone, embedding_model = build_model()

    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              class_weight=class_weight, verbose=2)

    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=cbs, class_weight=class_weight, verbose=2)

    return model, embedding_model


def fit_ensemble(embeddings_train, y_train):
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight="balanced")
    xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                         random_state=SEED, eval_metric="mlogloss")
    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)

    ensemble = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb), ("svm", svm)],
        voting="soft",
    )
    ensemble.fit(embeddings_train, y_train)
    return ensemble


# ---------------------------------------------------------------------------
# 3. Cross-validation driver
# ---------------------------------------------------------------------------
def main():
    print("Loading + preprocessing dataset (mask-guided crop + resize/normalize)...")
    X, y, pids = load_and_preprocess_dataset(DATA_DIR)
    print(f"Loaded {len(X)} slices from {len(set(pids))} patients")
    gc.collect()

    gkf = GroupKFold(n_splits=N_FOLDS)

    fold_results = {"cnn": [], "cnn_tta": [], "ensemble": []}
    all_val_frac = 0.15

    for fold_idx, (trainval_idx, test_idx) in enumerate(gkf.split(X, y, groups=pids)):
        print(f"\n{'='*70}\nFOLD {fold_idx + 1}/{N_FOLDS}\n{'='*70}")

        gss = GroupShuffleSplit(n_splits=1, test_size=all_val_frac, random_state=SEED)
        tr_sub, val_sub = next(gss.split(
            trainval_idx, y[trainval_idx], groups=pids[trainval_idx]
        ))
        train_idx = trainval_idx[tr_sub]
        val_idx = trainval_idx[val_sub]

        assert set(pids[train_idx]).isdisjoint(pids[test_idx])
        assert set(pids[val_idx]).isdisjoint(pids[test_idx])
        assert set(pids[train_idx]).isdisjoint(pids[val_idx])

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

        class_weight = dict(enumerate(
            compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
        ))
        augmenter = make_augmenter()

        model, embedding_model = train_fold_model(
            X_train, y_train, X_val, y_val, class_weight, augmenter
        )

        # --- CNN softmax, single pass ---
        y_pred_cnn = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
        acc_cnn = accuracy_score(y_test, y_pred_cnn)
        fold_results["cnn"].append(acc_cnn)

        # --- CNN softmax + TTA ---
        probs_tta = tta_predict(model, augmenter, X_test)
        y_pred_tta = np.argmax(probs_tta, axis=1)
        acc_tta = accuracy_score(y_test, y_pred_tta)
        fold_results["cnn_tta"].append(acc_tta)

        # --- Embedding ensemble (RF + XGBoost + SVM) ---
        emb_train = embedding_model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
        emb_test = embedding_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
        ensemble = fit_ensemble(emb_train, y_train)
        y_pred_ens = ensemble.predict(emb_test)
        acc_ens = accuracy_score(y_test, y_pred_ens)
        fold_results["ensemble"].append(acc_ens)

        print(f"\nFold {fold_idx + 1} results: CNN={acc_cnn:.4f}  CNN+TTA={acc_tta:.4f}  Ensemble={acc_ens:.4f}")
        print("\n-- CNN+TTA classification report --")
        print(classification_report(y_test, y_pred_tta, target_names=list(LABEL_MAP.values())))
        print(confusion_matrix(y_test, y_pred_tta))
        print("\n-- Embedding ensemble classification report --")
        print(classification_report(y_test, y_pred_ens, target_names=list(LABEL_MAP.values())))
        print(confusion_matrix(y_test, y_pred_ens))

        del model, embedding_model, ensemble, emb_train, emb_test
        tf.keras.backend.clear_session()
        gc.collect()

    print(f"\n{'='*70}\nFINAL 5-FOLD CROSS-VALIDATION SUMMARY\n{'='*70}")
    for key, label in [("cnn", "CNN (softmax)"), ("cnn_tta", "CNN + TTA"), ("ensemble", "CNN-embedding ensemble (RF+XGB+SVM)")]:
        accs = np.array(fold_results[key])
        print(f"{label:40s}: {accs.mean()*100:.2f}% +/- {accs.std()*100:.2f}%   (folds: {[f'{a*100:.2f}' for a in accs]})")


if __name__ == "__main__":
    main()


Loading + preprocessing dataset (mask-guided crop + resize/normalize)...


I0000 00:00:1790138583.625012      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790138583.628097      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  processed 500/3064
  processed 1000/3064
  processed 1500/3064
  processed 2000/3064
  processed 2500/3064
  processed 3000/3064
  processed 3064/3064
Loaded 3064 slices from 233 patients

FOLD 1/5
Train: 2012  Val: 439  Test: 613
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/8


2026-09-23 04:44:17.292206: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 04:44:17.433069: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 04:44:17.758295: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 04:44:17.900359: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 04:44:18.708300: E external/local_xla/xla/stream_

126/126 - 73s - 582ms/step - accuracy: 0.7087 - loss: 0.7871 - val_accuracy: 0.8679 - val_loss: 0.3427
Epoch 2/8
126/126 - 13s - 100ms/step - accuracy: 0.8052 - loss: 0.5226 - val_accuracy: 0.9112 - val_loss: 0.2573
Epoch 3/8
126/126 - 13s - 102ms/step - accuracy: 0.8101 - loss: 0.5014 - val_accuracy: 0.9134 - val_loss: 0.2337
Epoch 4/8
126/126 - 13s - 102ms/step - accuracy: 0.8260 - loss: 0.4410 - val_accuracy: 0.9362 - val_loss: 0.2163
Epoch 5/8
126/126 - 13s - 102ms/step - accuracy: 0.8424 - loss: 0.4112 - val_accuracy: 0.9294 - val_loss: 0.2147
Epoch 6/8
126/126 - 13s - 102ms/step - accuracy: 0.8330 - loss: 0.4361 - val_accuracy: 0.9021 - val_loss: 0.2471
Epoch 7/8
126/126 - 13s - 103ms/step - accuracy: 0.8459 - loss: 0.4084 - val_accuracy: 0.9248 - val_loss: 0.2278
Epoch 8/8
126/126 - 13s - 101ms/step - accuracy: 0.8439 - loss: 0.4056 - val_accuracy: 0.9157 - val_loss: 0.2466
Epoch 1/35


...

135/135 - 118s - 876ms/step - accuracy: 0.5966 - loss: 1.1270 - val_accuracy: 0.8249 - val_loss: 0.4682 - learning_rate: 1.0000e-05
Epoch 2/35
135/135 - 14s - 103ms/step - accuracy: 0.6857 - loss: 0.8618 - val_accuracy: 0.8350 - val_loss: 0.4396 - learning_rate: 1.0000e-05
Epoch 3/35
135/135 - 14s - 100ms/step - accuracy: 0.7289 - loss: 0.7442 - val_accuracy: 0.8283 - val_loss: 0.4068 - learning_rate: 1.0000e-05
Epoch 4/35
135/135 - 14s - 101ms/step - accuracy: 0.7275 - loss: 0.7282 - val_accuracy: 0.8552 - val_loss: 0.3752 - learning_rate: 1.0000e-05
Epoch 5/35
135/135 - 15s - 108ms/step - accuracy: 0.7721 - loss: 0.5915 - val_accuracy: 0.8552 - val_loss: 0.3567 - learning_rate: 1.0000e-05
Epoch 6/35
135/135 - 14s - 101ms/step - accuracy: 0.7711 - loss: 0.5642 - val_accuracy: 0.8653 - val_loss: 0.3306 - learning_rate: 1.0000e-05
Epoch 7/35
135/135 - 14s - 102ms/step - accuracy: 0.7934 - loss: 0.5373 - val_accuracy: 0.8687 - val_loss: 0.3172 - learning_rate: 1.0000e-05
Epoch 8/35
135/1

2026-09-23 05:38:40.426490: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:38:40.563354: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:38:40.879020: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:38:41.020440: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:38:41.781597: E external/local_xla/xla/stream_

130/130 - 60s - 460ms/step - accuracy: 0.7172 - loss: 0.8014 - val_accuracy: 0.8511 - val_loss: 0.4358
Epoch 2/8
130/130 - 13s - 100ms/step - accuracy: 0.7828 - loss: 0.5888 - val_accuracy: 0.8484 - val_loss: 0.3789
Epoch 3/8
130/130 - 13s - 100ms/step - accuracy: 0.8025 - loss: 0.5191 - val_accuracy: 0.8644 - val_loss: 0.3260
Epoch 4/8
130/130 - 13s - 99ms/step - accuracy: 0.8001 - loss: 0.5059 - val_accuracy: 0.8803 - val_loss: 0.3074
Epoch 5/8
130/130 - 13s - 99ms/step - accuracy: 0.8353 - loss: 0.4550 - val_accuracy: 0.8777 - val_loss: 0.3114
Epoch 6/8
130/130 - 13s - 98ms/step - accuracy: 0.8333 - loss: 0.4146 - val_accuracy: 0.8697 - val_loss: 0.3170
Epoch 7/8
130/130 - 13s - 98ms/step - accuracy: 0.8353 - loss: 0.4347 - val_accuracy: 0.8936 - val_loss: 0.2908
Epoch 8/8
130/130 - 13s - 99ms/step - accuracy: 0.8285 - loss: 0.4359 - val_accuracy: 0.8830 - val_loss: 0.2860
Epoch 1/35
130/130 - 112s - 863ms/step - accuracy: 0.5371 - loss: 1.1726 - val_accuracy: 0.8059 - val_loss: 0.4

2026-09-23 05:49:56.990690: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:49:57.125823: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:49:57.444393: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:49:57.585509: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 05:49:58.367248: E external/local_xla/xla/stream_


Fold 5 results: CNN=0.9183  CNN+TTA=0.9216  Ensemble=0.9346

-- CNN+TTA classification report --
              precision    recall  f1-score   support

  meningioma       0.82      0.92      0.86       125
      glioma       0.99      0.93      0.96       403
   pituitary       0.80      0.90      0.85        84

    accuracy                           0.92       612
   macro avg       0.87      0.92      0.89       612
weighted avg       0.93      0.92      0.92       612

[[115   1   9]
 [ 20 373  10]
 [  6   2  76]]

-- Embedding ensemble classification report --
              precision    recall  f1-score   support

  meningioma       0.88      0.90      0.89       125
      glioma       0.99      0.94      0.96       403
   pituitary       0.80      0.95      0.87        84

    accuracy                           0.93       612
   macro avg       0.89      0.93      0.91       612
weighted avg       0.94      0.93      0.94       612

[[112   3  10]
 [ 13 380  10]
 [  2   2  80]]


In [3]:
import os
for split in ["Training", "Testing"]:
    p = f"/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/{split}"
    print(split, os.listdir(p) if os.path.exists(p) else "NOT FOUND")

Training ['pituitary', 'notumor', 'meningioma', 'glioma']
Testing ['pituitary', 'notumor', 'meningioma', 'glioma']


In [4]:
"""
Step 4: 4-class Kaggle "Brain Tumor MRI Dataset" (masoudnickparvar) benchmark.

Dataset: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset
Classes: glioma, meningioma, notumor, pituitary
Structure (standard Kaggle release):
    brain-tumor-mri-dataset/
        Training/
            glioma/*.jpg
            meningioma/*.jpg
            notumor/*.jpg
            pituitary/*.jpg
        Testing/
            glioma/*.jpg
            meningioma/*.jpg
            notumor/*.jpg
            pituitary/*.jpg

IMPORTANT LIMITATION vs. the figshare pipeline (step3_cv_ensemble.py):
  - This dataset provides no tumor segmentation masks -> no mask-guided ROI
    cropping here. The whole slice is used as-is (center-cropped/resized).
  - This dataset provides no patient ID field -> cross-validation here is
    standard stratified k-fold over images, NOT patient-level. If you find
    documentation confirming no patient overlap between Training/Testing (or
    within-class duplicates), note that in the paper; otherwise disclose this
    as a limitation, since some sources reused for this merged dataset are
    known to contain near-duplicate slices from the same scan.

Otherwise reuses the same recipe as step3: EfficientNetB0 backbone, two-stage
fine-tuning, class weighting, TTA, 5-fold CV, CNN-embedding ensemble
(RF + XGBoost + SVM).
"""

import os
import gc
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 35
FINE_TUNE_AT = 0.8
TTA_ROUNDS = 5
N_FOLDS = 5
SEED = 42
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASS_NAMES)


# ---------------------------------------------------------------------------
# 1. Load + preprocess: stream each file straight into a uint8 array
#    (same memory-safety approach as step3, to avoid Kaggle OOM restarts)
# ---------------------------------------------------------------------------
def list_image_paths(data_dir):
    """Combine Training/ and Testing/ folders -- we re-split ourselves via
    k-fold rather than relying on the dataset's original train/test split,
    since we want cross-validated, not single-split, accuracy."""
    paths, labels = [], []
    for split_dir in ["Training", "Testing"]:
        for class_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = os.path.join(data_dir, split_dir, class_name)
            files = sorted(glob.glob(os.path.join(class_dir, "*")))
            paths.extend(files)
            labels.extend([class_idx] * len(files))

    if not paths:
        raise FileNotFoundError(
            f"No images found under {data_dir}/Training or /Testing. "
            f"Check DATA_DIR and folder names match {CLASS_NAMES}."
        )
    return paths, np.array(labels)


def load_and_preprocess(paths):
    n = len(paths)
    X = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        img = tf.io.read_file(p)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        X[i] = img.numpy().astype(np.uint8)
        if (i + 1) % 1000 == 0 or (i + 1) == n:
            print(f"  processed {i + 1}/{n}")
    return X


# ---------------------------------------------------------------------------
# 2. Model: EfficientNetB0 backbone, with an embedding-extraction hook
# ---------------------------------------------------------------------------
def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    embedding = layers.Dense(128, activation="relu", name="embedding")(x)
    x = layers.BatchNormalization()(embedding)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    embedding_model = models.Model(inputs, embedding)
    return model, backbone, embedding_model


def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])


def tta_predict(model, augmenter, X, rounds=TTA_ROUNDS, batch_size=BATCH_SIZE):
    probs = np.zeros((X.shape[0], NUM_CLASSES), dtype=np.float32)
    probs += model.predict(X, batch_size=batch_size, verbose=0)
    for _ in range(rounds):
        X_aug = augmenter(X, training=True).numpy()
        probs += model.predict(X_aug, batch_size=batch_size, verbose=0)
    probs /= (rounds + 1)
    return probs


def train_fold_model(X_train, y_train, X_val, y_val, class_weight, augmenter):
    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model, backbone, embedding_model = build_model()

    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              class_weight=class_weight, verbose=2)

    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=cbs, class_weight=class_weight, verbose=2)

    return model, embedding_model


def fit_ensemble(embeddings_train, y_train):
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight="balanced")
    xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                         random_state=SEED, eval_metric="mlogloss")
    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)

    ensemble = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb), ("svm", svm)],
        voting="soft",
    )
    ensemble.fit(embeddings_train, y_train)
    return ensemble


# ---------------------------------------------------------------------------
# 3. Cross-validation driver
# ---------------------------------------------------------------------------
def main():
    print("Listing image paths...")
    paths, y = list_image_paths(DATA_DIR)
    print(f"Found {len(paths)} images across {NUM_CLASSES} classes: {CLASS_NAMES}")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {(y == i).sum()}")

    print("\nLoading + preprocessing images...")
    X = load_and_preprocess(paths)
    gc.collect()

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_results = {"cnn": [], "cnn_tta": [], "ensemble": []}

    for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*70}\nFOLD {fold_idx + 1}/{N_FOLDS}\n{'='*70}")

        train_idx, val_idx = train_test_split(
            trainval_idx, test_size=0.15, random_state=SEED, stratify=y[trainval_idx]
        )

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

        class_weight = dict(enumerate(
            compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
        ))
        augmenter = make_augmenter()

        model, embedding_model = train_fold_model(
            X_train, y_train, X_val, y_val, class_weight, augmenter
        )

        y_pred_cnn = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
        acc_cnn = accuracy_score(y_test, y_pred_cnn)
        fold_results["cnn"].append(acc_cnn)

        probs_tta = tta_predict(model, augmenter, X_test)
        y_pred_tta = np.argmax(probs_tta, axis=1)
        acc_tta = accuracy_score(y_test, y_pred_tta)
        fold_results["cnn_tta"].append(acc_tta)

        emb_train = embedding_model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
        emb_test = embedding_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
        ensemble = fit_ensemble(emb_train, y_train)
        y_pred_ens = ensemble.predict(emb_test)
        acc_ens = accuracy_score(y_test, y_pred_ens)
        fold_results["ensemble"].append(acc_ens)

        print(f"\nFold {fold_idx + 1} results: CNN={acc_cnn:.4f}  CNN+TTA={acc_tta:.4f}  Ensemble={acc_ens:.4f}")
        print("\n-- CNN+TTA classification report --")
        print(classification_report(y_test, y_pred_tta, target_names=CLASS_NAMES))
        print(confusion_matrix(y_test, y_pred_tta))
        print("\n-- Embedding ensemble classification report --")
        print(classification_report(y_test, y_pred_ens, target_names=CLASS_NAMES))
        print(confusion_matrix(y_test, y_pred_ens))

        del model, embedding_model, ensemble, emb_train, emb_test
        tf.keras.backend.clear_session()
        gc.collect()

    print(f"\n{'='*70}\nFINAL 5-FOLD CROSS-VALIDATION SUMMARY (4-class Kaggle dataset)\n{'='*70}")
    for key, label in [("cnn", "CNN (softmax)"), ("cnn_tta", "CNN + TTA"), ("ensemble", "CNN-embedding ensemble (RF+XGB+SVM)")]:
        accs = np.array(fold_results[key])
        print(f"{label:40s}: {accs.mean()*100:.2f}% +/- {accs.std()*100:.2f}%   (folds: {[f'{a*100:.2f}' for a in accs]})")


if __name__ == "__main__":
    main()


Listing image paths...
Found 7200 images across 4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
  glioma: 1800
  meningioma: 1800
  notumor: 1800
  pituitary: 1800

Loading + preprocessing images...
  processed 1000/7200
  processed 2000/7200
  processed 3000/7200
  processed 4000/7200
  processed 5000/7200
  processed 6000/7200
  processed 7000/7200
  processed 7200/7200

FOLD 1/5
Train: 4896  Val: 864  Test: 1440
Epoch 1/8
306/306 - 57s - 187ms/step - accuracy: 0.7382 - loss: 0.7545 - val_accuracy: 0.7280 - val_loss: 0.7105
Epoch 2/8
306/306 - 30s - 98ms/step - accuracy: 0.8078 - loss: 0.5124 - val_accuracy: 0.7894 - val_loss: 0.5433
Epoch 3/8
306/306 - 29s - 96ms/step - accuracy: 0.8270 - loss: 0.4550 - val_accuracy: 0.8171 - val_loss: 0.5090
Epoch 4/8
306/306 - 30s - 97ms/step - accuracy: 0.8356 - loss: 0.4422 - val_accuracy: 0.8137 - val_loss: 0.4565
Epoch 5/8
306/306 - 29s - 95ms/step - accuracy: 0.8409 - loss: 0.4214 - val_accuracy: 0.8322 - val_loss: 0.4356
Epoch 6/

In [1]:
"""
Step 5: Grad-CAM visualization for the trained EfficientNetB0 brain tumor
classifier, for interpretability figures in the paper.

Trains one model (same recipe as step4_kaggle4class.py, single fold -- no
need to redo the full 5-fold CV for this) on the 4-class dataset, then
generates Grad-CAM heatmaps overlaid on sample test images from each class,
showing which regions of the MRI the model attended to when making its
prediction.

Run on Kaggle (GPU enabled). Outputs PNGs to /kaggle/working/gradcam/ --
download these and place them in images/ for the paper.
"""

import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 35
FINE_TUNE_AT = 0.8
SEED = 42
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASS_NAMES)
OUT_DIR = "/kaggle/working/gradcam"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------------
# 1. Data loading (same as step4)
# ---------------------------------------------------------------------------
def list_image_paths(data_dir):
    paths, labels = [], []
    for split_dir in ["Training", "Testing"]:
        for class_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = os.path.join(data_dir, split_dir, class_name)
            files = sorted(glob.glob(os.path.join(class_dir, "*")))
            paths.extend(files)
            labels.extend([class_idx] * len(files))
    return paths, np.array(labels)


def load_and_preprocess(paths):
    n = len(paths)
    X = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        img = tf.io.read_file(p)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        X[i] = img.numpy().astype(np.uint8)
        if (i + 1) % 1000 == 0 or (i + 1) == n:
            print(f"  processed {i + 1}/{n}")
    return X


# ---------------------------------------------------------------------------
# 2. Model (same architecture as step4, but we need direct access to the
#    backbone submodel to locate the last conv layer for Grad-CAM)
# ---------------------------------------------------------------------------
def build_and_train_model(X_train, y_train, X_val, y_val, class_weight):
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    conv_features = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(conv_features)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)

    augmenter = tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.1),
    ])

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
                .prefetch(tf.data.AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
              .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

    model.compile(optimizer=optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              class_weight=class_weight, verbose=2)

    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * (1 - FINE_TUNE_AT))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(1e-5),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=cbs, class_weight=class_weight, verbose=2)

    return model, backbone


# ---------------------------------------------------------------------------
# 3. Grad-CAM
# ---------------------------------------------------------------------------
def make_gradcam_heatmap(img_array, model, backbone, pred_index=None):
    """Standard Grad-CAM: gradient of the predicted class score w.r.t. the
    backbone's last conv feature map, global-average-pooled into per-channel
    weights, then a weighted sum of the feature maps -> a coarse localization
    heatmap showing which spatial regions drove the prediction.

    NOTE: building a second Model that reaches into
    `backbone.get_layer(...).output` fails in Keras 3 when the backbone was
    called as a single nested layer inside the outer model (it raises a
    KeyError -- the backbone's internal tensors aren't part of the outer
    model's graph). Instead, we replay the head layers manually inside the
    GradientTape, on top of the backbone's own output tensor -- for
    EfficientNetB0 with include_top=False, that output IS already the final
    conv feature map (post top_conv/top_bn/top_activation), so no need to
    dig into a named internal layer at all.
    """
    backbone_idx = model.layers.index(backbone)
    pre_layers = model.layers[1:backbone_idx]   # skip InputLayer, keep preprocessing op(s)
    post_layers = model.layers[backbone_idx + 1:]  # GAP, BN, Dropout, Dense, ... softmax

    x = img_array
    for layer in pre_layers:
        x = layer(x)

    with tf.GradientTape() as tape:
        conv_output = backbone(x, training=False)
        tape.watch(conv_output)
        h = conv_output
        for layer in post_layers:
            h = layer(h, training=False)
        predictions = h
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), predictions.numpy()[0]


def overlay_and_save(orig_img, heatmap, save_path, alpha=0.4):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    jet_heatmap = np.uint8(jet_heatmap * 255)

    superimposed = jet_heatmap * alpha + orig_img * (1 - alpha)
    superimposed = np.uint8(superimposed)

    _, axes = plt.subplots(1, 3, figsize=(9, 3.2))
    axes[0].imshow(orig_img.astype(np.uint8))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(heatmap_resized, cmap="jet")
    axes[1].set_title("Grad-CAM Heatmap")
    axes[1].axis("off")
    axes[2].imshow(superimposed)
    axes[2].set_title("Overlay")
    axes[2].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# ---------------------------------------------------------------------------
# 4. Driver
# ---------------------------------------------------------------------------
def main():
    print("Listing image paths...")
    paths, y = list_image_paths(DATA_DIR)
    print(f"Found {len(paths)} images across {NUM_CLASSES} classes: {CLASS_NAMES}")

    print("Loading + preprocessing images...")
    X = load_and_preprocess(paths)

    trainval_idx, test_idx = train_test_split(
        np.arange(len(X)), test_size=0.2, random_state=SEED, stratify=y
    )
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=0.15, random_state=SEED, stratify=y[trainval_idx]
    )

    class_weight = dict(enumerate(
        compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y[train_idx])
    ))

    print("Training model for Grad-CAM visualization (single split, not full CV)...")
    model, backbone = build_and_train_model(
        X[train_idx], y[train_idx], X[val_idx], y[val_idx], class_weight
    )

    print(f"\nGenerating Grad-CAM visualizations, saving to {OUT_DIR} ...")
    rng = np.random.RandomState(SEED)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_test_idx = test_idx[y[test_idx] == class_idx]
        chosen = rng.choice(class_test_idx, size=min(3, len(class_test_idx)), replace=False)
        for i, idx in enumerate(chosen):
            img = X[idx]
            img_array = np.expand_dims(img, axis=0).astype(np.float32)
            heatmap, pred_class, pred_probs = make_gradcam_heatmap(
                img_array, model, backbone
            )
            correct = "correct" if pred_class == class_idx else "MISCLASSIFIED"
            save_path = os.path.join(
                OUT_DIR, f"gradcam_{class_name}_{i}_{correct}.png"
            )
            overlay_and_save(img, heatmap, save_path)
            print(f"  {class_name} sample {i}: predicted={CLASS_NAMES[pred_class]} "
                  f"(confidence {pred_probs[pred_class]:.2f}) -- {correct}")

    print(f"\nDone. Download the images in {OUT_DIR} and place the best "
          f"representative ones (e.g. one per class) into images/ for the paper.")


if __name__ == "__main__":
    main()


Listing image paths...
Found 7200 images across 4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Loading + preprocessing images...


I0000 00:00:1790225874.706127      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790225874.709206      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  processed 1000/7200
  processed 2000/7200
  processed 3000/7200
  processed 4000/7200
  processed 5000/7200
  processed 6000/7200
  processed 7000/7200
  processed 7200/7200
Training model for Grad-CAM visualization (single split, not full CV)...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8


2026-09-24 04:59:32.986169: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 04:59:33.126485: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 04:59:33.449650: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 04:59:33.591423: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 04:59:33.740539: E external/local_xla/xla/stream_

306/306 - 66s - 215ms/step - accuracy: 0.7388 - loss: 0.7408 - val_accuracy: 0.7731 - val_loss: 0.6264
Epoch 2/8
306/306 - 30s - 99ms/step - accuracy: 0.8100 - loss: 0.5024 - val_accuracy: 0.8472 - val_loss: 0.4224
Epoch 3/8
306/306 - 31s - 100ms/step - accuracy: 0.8288 - loss: 0.4675 - val_accuracy: 0.8403 - val_loss: 0.4350
Epoch 4/8
306/306 - 30s - 98ms/step - accuracy: 0.8384 - loss: 0.4258 - val_accuracy: 0.8287 - val_loss: 0.4555
Epoch 5/8
306/306 - 30s - 98ms/step - accuracy: 0.8458 - loss: 0.4136 - val_accuracy: 0.8530 - val_loss: 0.3886
Epoch 6/8
306/306 - 30s - 99ms/step - accuracy: 0.8513 - loss: 0.4031 - val_accuracy: 0.8322 - val_loss: 0.4499
Epoch 7/8
306/306 - 30s - 98ms/step - accuracy: 0.8489 - loss: 0.3926 - val_accuracy: 0.8495 - val_loss: 0.4181
Epoch 8/8
306/306 - 30s - 98ms/step - accuracy: 0.8601 - loss: 0.3782 - val_accuracy: 0.8611 - val_loss: 0.3599
Epoch 1/35


2026-09-24 05:04:29.148342: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-24 05:04:29.290600: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


306/306 - 99s - 324ms/step - accuracy: 0.6761 - loss: 0.9251 - val_accuracy: 0.7847 - val_loss: 0.5889 - learning_rate: 1.0000e-05
Epoch 2/35
306/306 - 30s - 99ms/step - accuracy: 0.7484 - loss: 0.6782 - val_accuracy: 0.8171 - val_loss: 0.5264 - learning_rate: 1.0000e-05
Epoch 3/35
306/306 - 30s - 99ms/step - accuracy: 0.7980 - loss: 0.5493 - val_accuracy: 0.8252 - val_loss: 0.4897 - learning_rate: 1.0000e-05
Epoch 4/35
306/306 - 30s - 99ms/step - accuracy: 0.8074 - loss: 0.5202 - val_accuracy: 0.8438 - val_loss: 0.4395 - learning_rate: 1.0000e-05
Epoch 5/35
306/306 - 30s - 99ms/step - accuracy: 0.8303 - loss: 0.4678 - val_accuracy: 0.8472 - val_loss: 0.4110 - learning_rate: 1.0000e-05
Epoch 6/35
306/306 - 30s - 98ms/step - accuracy: 0.8472 - loss: 0.4236 - val_accuracy: 0.8519 - val_loss: 0.3786 - learning_rate: 1.0000e-05
Epoch 7/35
306/306 - 30s - 99ms/step - accuracy: 0.8597 - loss: 0.3968 - val_accuracy: 0.8611 - val_loss: 0.3455 - learning_rate: 1.0000e-05
Epoch 8/35
306/306 - 30

/tmp/ipykernel_58/199658945.py:183: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


  glioma sample 0: predicted=glioma (confidence 1.00) -- correct
  glioma sample 1: predicted=glioma (confidence 0.93) -- correct
  glioma sample 2: predicted=glioma (confidence 1.00) -- correct
  meningioma sample 0: predicted=meningioma (confidence 1.00) -- correct
  meningioma sample 1: predicted=meningioma (confidence 0.99) -- correct
  meningioma sample 2: predicted=meningioma (confidence 0.99) -- correct
  notumor sample 0: predicted=notumor (confidence 1.00) -- correct
  notumor sample 1: predicted=notumor (confidence 1.00) -- correct
  notumor sample 2: predicted=notumor (confidence 1.00) -- correct
  pituitary sample 0: predicted=pituitary (confidence 1.00) -- correct
  pituitary sample 1: predicted=pituitary (confidence 1.00) -- correct
  pituitary sample 2: predicted=pituitary (confidence 1.00) -- correct

Done. Download the images in /kaggle/working/gradcam and place the best representative ones (e.g. one per class) into images/ for the paper.


In [1]:
"""
Run this on Kaggle (same notebook/session where the brain-tumor-mri-dataset
is already mounted) to build the sample-slice grid figure for the paper's
Dataset section (images/dataset_image.PNG).

Grabs one representative Training image per class and arranges them in a
1x4 grid with class labels. Download the resulting PNG and place it at
images/dataset_image.PNG in the paper repo.
"""

import os
import glob
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATA_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
OUT_PATH = "/kaggle/working/dataset_image.PNG"

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, class_name in zip(axes, CLASS_NAMES):
    class_dir = os.path.join(DATA_DIR, "Training", class_name)
    sample_path = sorted(glob.glob(os.path.join(class_dir, "*")))[0]
    img = mpimg.imread(sample_path)
    ax.imshow(img, cmap="gray")
    ax.set_title(class_name.capitalize(), fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUT_PATH, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved {OUT_PATH} -- download this and place it at images/dataset_image.PNG")


Saved /kaggle/working/dataset_image.PNG -- download this and place it at images/dataset_image.PNG
